# RAG Application Example - Code Analysis

This notebook demonstrates a real-world RAG application for analyzing code files.
The example shows how to query code repositories and get intelligent responses about code patterns, best practices, and potential issues.

In [43]:
from dotenv import load_dotenv

load_dotenv(dotenv_path='.env')

True

In [44]:
from llama_index.llms.nvidia import NVIDIA
from llama_index.core import Settings

# Initialize with an NVIDIA model (e.g., Llama 3)
llm = NVIDIA(model="openai/gpt-oss-120b")
Settings.llm = llm

print(f"Using LLM: {llm.model}")

Using LLM: openai/gpt-oss-120b


In [45]:
from llama_index.embeddings.nvidia import NVIDIAEmbedding
from llama_index.core import Settings

embedder = NVIDIAEmbedding(model="nvidia/nv-embedqa-e5-v5")
Settings.embed_model = embedder

print(f"Using Embedding model: {embedder.model}")

Using Embedding model: nvidia/nv-embedqa-e5-v5


In [46]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.nvidia import NVIDIA
from llama_index.embeddings.nvidia import NVIDIAEmbedding
from llama_index.core.text_splitter import TokenTextSplitter

# Load environment variables
load_dotenv(dotenv_path='.env')

# Initialize LLM with NVIDIA model
llm = NVIDIA(model="openai/gpt-oss-120b")

# Initialize embedding model
embedder = NVIDIAEmbedding(model="nvidia/nv-embedqa-e5-v5")

# Configure settings
Settings.llm = llm
Settings.embed_model = embedder

# Load documents from the data directory
documents = SimpleDirectoryReader("data").load_data()
print(f"Loaded {len(documents)} documents")

# Chunk documents to ensure they fit within embedding model limits
# NVIDIA embedqa-e5-v5 has a 512 token limit
text_splitter = TokenTextSplitter(
    chunk_size=200,          # Smaller chunks improve retrieval precision
    chunk_overlap=40         # Keep overlap so context is preserved
)

nodes = text_splitter.get_nodes_from_documents(documents)
print(f"Split into {len(nodes)} document chunks (max {text_splitter.chunk_size} tokens each)")

# Create vector index from chunked document nodes
index = VectorStoreIndex(nodes, embed_model=embedder)
query_engine = index.as_query_engine(
    similarity_top_k=7,
    response_mode="compact",
)

Loaded 7 documents
Split into 9 document chunks (max 200 tokens each)


## Example Queries

In [47]:
# Example 1: General knowledge from documents
query = "What are the main features of this project?"
response = query_engine.query(query)
print(f"Query: {query}")
print(f"\nResponse:\n{response}")

Query: What are the main features of this project?

Response:
The project provides a complete Retrieval‑Augmented Generation pipeline built around local document ingestion and NVIDIA‑powered embeddings. It reads files from a directory with SimpleDirectoryReader, splits the text into manageable chunks using TokenTextSplitter (typically chunk_size ≈ 150 with chunk_overlap ≈ 30), and embeds each chunk with the NVIDIA nv‑embedqa‑e5‑v5 model. The embeddings are stored in a VectorStoreIndex, enabling fast similarity search. A query engine is then created (often with similarity_top_k = 7 and response_mode = "compact") to retrieve the most relevant nodes and generate concise, project‑specific answers. The setup relies on uv for reproducible dependency installation, python‑dotenv for managing the NVIDIA_API_KEY, and integrates the llama‑index and langchain‑nvidia libraries to tie everything together.


In [48]:
# Example 2: Code implementation questions
query = "How do I implement a Retrieval Augmented Generation system with Python?"
response = query_engine.query(query)
print(f"Query: {query}")
print(f"\nResponse:\n{response}")

Query: How do I implement a Retrieval Augmented Generation system with Python?

Response:
Implement the RAG pipeline in Python by following these steps:

1. **Read the source files**  
   ```python
   from llama_index import SimpleDirectoryReader
   documents = SimpleDirectoryReader("data").load_data()
   ```

2. **Chunk the text** – keep each chunk under the model’s token limit and add overlap so the query engine can reconstruct context:  
   ```python
   from llama_index import TokenTextSplitter
   splitter = TokenTextSplitter(chunk_size=150, chunk_overlap=30)
   nodes = splitter.split_documents(documents)
   ```

3. **Embed the chunks** – use the NVIDIA embedding model (`nvidia/nv-embedqa-e5-v5`) via the Llama‑Index NVIDIA embeddings integration:  
   ```python
   from llama_index.embeddings.nvidia import NvidiaEmbedding
   embed_model = NvidiaEmbedding()
   ```

4. **Build a vector store index** – store the embeddings for fast similarity search:  
   ```python
   from llama_index i

In [49]:
# Example 3: Dependency questions
query = "What dependencies are used for the RAG pipeline?"
response = query_engine.query(query)
print(f"Query: {query}")
print(f"\nResponse:\n{response}")

Query: What dependencies are used for the RAG pipeline?

Response:
The RAG pipeline depends on the following packages:

- `jupyterlab` (≥ 4.5.8)  
- `langchain` (≥ 1.3.4)  
- `langchain‑nvidia‑ai‑endpoints` (≥ 1.4.1)  
- `langchain‑nvidia‑langgraph` (installed from GitHub)  
- `llama‑index‑core` (≥ 0.14.3, < 0.15)  
- `llama‑index‑embeddings‑nvidia` (≥ 0.5.1, < 0.6)  
- `llama‑index‑llms‑nvidia` (≥ 0.5.0, < 0.6)  
- `llama‑index‑multi‑modal‑llms‑nvidia` (≥ 0.5.2, < 0.6)  
- `python‑dotenv` (≥ 1.2.2)


In [50]:
# Example 4: Installation and setup questions
query = "How do I install and configure this project?"
response = query_engine.query(query)
print(f"Query: {query}")
print(f"\nResponse:\n{response}")

Query: How do I install and configure this project?

Response:
1. Create and activate the project’s virtual environment (the repository includes a `.venv` folder).  
2. Inside the activated environment run the reproducible installer:

```bash
uv sync --active
```

   This reads the `pyproject.toml` and installs the exact versions of all required packages (JupyterLab, LangChain, the NVIDIA endpoint libraries, the Llama‑Index components, python‑dotenv, etc.).

3. Add your NVIDIA API key to the environment file:

```text
# .env (in the project root)
NVIDIA_API_KEY=your_key_here
```

4. Launch the notebook that drives the Retrieval‑Augmented Generation example:

```bash
jupyter lab rag_example.ipynb
```

5. Run the cells in the notebook. The first steps will load documents, split them with `TokenTextSplitter`, embed them using the `nvidia/nv-embedqa-e5-v5` model, build a `VectorStoreIndex`, and set up the query engine.

After these steps the project is installed and ready for use.
